In [ ]:
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

: 

In [ ]:
dataset = "../Dataset/Crop_recommendation.csv"
df = pd.read_csv(dataset)
df = df.dropna(axis=0)
print(df.info())
print(df.shape)
print(df.columns)
print(df.select_dtypes(include=["object"]).columns)

In [ ]:
string_columns = df.select_dtypes(["object"]).columns
def encode_data():
    encoders = {}
    for i in string_columns:
        encoder = LabelEncoder()
        df[i] = encoder.fit_transform(df[i])
        encoders[i] = encoder
        
    return encoders
    
target_values = {}
encoders = encode_data()
print("\nEncoding Clases\n")
for y in string_columns:
    for i in encoders[y].classes_:
        target_values[i] = encoders[y].transform([i])[0]
        print(i,  encoders[y].transform([i]))
    print("\n")
print("\n")
print(encoders)

In [ ]:
X = df.drop(columns=["label"])
Y = df["label"]

xtrain, xtest, ytrain, ytest = train_test_split(X, Y, test_size=0.2, random_state=42)

print(f"[+] Training data amount: {len(xtrain)}")
print(f"[+] Testing data amount: {len(xtest)}")

In [ ]:
def evaluate(X, Y):
    model = RandomForestClassifier(random_state=42)
    params = {"n_estimators": [i for i in range(1, 20)],
          }

    grid = GridSearchCV(
        estimator=model,
        param_grid=params,
        cv=5
    )

    grid.fit(X, Y)
    
    best_params = grid.best_params_
    best_model = grid.best_estimator_
    
    cv_results = grid.cv_results_
    print("[+] Model Performance")
    count = 0
    params = cv_results["params"]
    for i in params:
        for key, value in i.items():
            print(f"Parameter {count + 1}. {key}: {value}")
            print("* Split 1 test Score:", round(cv_results["split0_test_score"][count] * 100, 2), "%")
            print("* Split 2 test Score:", round(cv_results["split1_test_score"][count] * 100, 2), "%")
            print("* Split 3 test Score:", round(cv_results["split2_test_score"][count] * 100, 2), "%")
            print("* Split 4 test Score:", round(cv_results["split3_test_score"][count] * 100, 2), "%")
            print("* Split 5 test Score:", round(cv_results["split4_test_score"][count] * 100, 2), "%")
            test_scores = [round(cv_results["split0_test_score"][count] * 100, 2), round(cv_results["split1_test_score"][count] * 100, 2), round(cv_results["split2_test_score"][count] * 100, 2), round(cv_results["split3_test_score"][count] * 100, 2), round(cv_results["split4_test_score"][count] * 100, 2)]
            print("* Average test Score:", round(np.mean(test_scores), 2), "%")
            count += 1
        print("\n")
            
    print(f"[+] Best parameters: ")
    for key, value in best_params.items():
        print(f"* {key}: {value}")
        
    train_predictions = best_model.predict(X)
    predictions = best_model.predict(xtest)
    train_acc = round(accuracy_score(Y, train_predictions)*100, 2)
    acc = round(accuracy_score(ytest, predictions)*100, 2)
    print(f"* Train data accuracy: {train_acc}%")
    print(f"* Test data accuracy: {acc}%")
    return best_model,predictions  
    
model,predictions = evaluate(xtrain, ytrain)
model

In [ ]:
plt.figure(figsize=(12, 10))
cm = confusion_matrix(ytest, predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)

In [ ]:
export_data = {
    "model": model,
    "encoder": encoders
}

joblib.dump(export_data, "model.pkl")